#### Import Library

In [36]:
# NOTE: json = read JSON
# NOTE: Path = safer file paths than raw strings
# NOTE: pandas = convert to table + export CSV

import json
from pathlib import Path
import pandas as pd

#### File path + existence check

In [37]:
# NOTE: put the json file in the same folder as your notebook,
# or change the path here

input_path = Path("easy_orders_practice_nulls.json")
input_path

WindowsPath('easy_orders_practice_nulls.json')

In [38]:
# NOTE: Always check the file exsists before reading, saves time.
print("Exists ?", input_path.exists)
print("Absoluate path: ", input_path.resolve())

Exists ? <bound method Path.exists of WindowsPath('easy_orders_practice_nulls.json')>
Absoluate path:  C:\Users\pdinh\Python\Python_Full_Course\Python Practice\JSON-Python\easy_orders_practice_nulls.json


#### Load JSON + inpsect the "Share"

In [39]:
# NOTE: Read the file text then parse JSON into Python Objects (dict/list)
transaction = json.loads(input_path.read_text(encoding="utf-8"))
transaction

{'meta': {'source': 'demo_api_practice', 'pulled_at': '2026-02-05T02:30:00Z'},
 'data': [{'order_id': 'ord_010',
   'order_time': '2026-02-05T09:05:10Z',
   'customer_id': 201,
   'city': 'Sydney',
   'total_amount': 22.75,
   'currency': 'AUD'},
  {'order_id': 'ord_011',
   'order_time': '2026-02-05T09:12:44Z',
   'customer_id': 202,
   'city': None,
   'total_amount': 15.0,
   'currency': 'AUD'},
  {'order_id': 'ord_012',
   'order_time': '2026-02-05T09:30:00Z',
   'customer_id': '203',
   'city': 'Brisbane',
   'total_amount': None,
   'currency': 'AUD'},
  {'order_id': 'ord_013',
   'order_time': 'bad_timestamp',
   'customer_id': 204,
   'city': 'Melbourne',
   'total_amount': 9.5,
   'currency': None},
  {'order_id': 'ord_014',
   'order_time': '2026-02-05T10:05:55Z',
   'customer_id': None,
   'city': 'Perth',
   'total_amount': 0,
   'currency': 'AUD'}]}

In [40]:
# NOTE: Confirm the top level structures
print("Top Level Keys: ", list(transaction.keys()))

# NOTE: meta is usually infor about the API pull, not row
print("Meta: ", transaction.get("meta", None))

# NOTE: data is where the rows usually are (list of dict records)
data = transaction.get("data",list())
print("Type of Data: ", type(data))
print("Number of records", len(data))

Top Level Keys:  ['meta', 'data']
Meta:  {'source': 'demo_api_practice', 'pulled_at': '2026-02-05T02:30:00Z'}
Type of Data:  <class 'list'>
Number of records 5


#### Convert to DataFrame (Table)

In [41]:
# NOTE: A list of dicts is the easiest case: directly become a table
df = pd.DataFrame(data)
df.head()

,order_id,order_time,customer_id,city,total_amount,currency
0,ord_010,2026-02-05T09:05:10Z,201,Sydney,22.75,AUD
1,ord_011,2026-02-05T09:12:44Z,202,NaN,15.00,AUD
2,ord_012,2026-02-05T09:30:00Z,203,Brisbane,NaN,AUD
3,ord_013,bad_timestamp,204,Melbourne,9.50,NaN
4,ord_014,2026-02-05T10:05:55Z,None,Perth,0.00,AUD


#### Clean data types 

##### Scan all columns to check N/A, duplicated and missing values

In [42]:
# NOTE: Check the data types for each column in the dataset
print("Dtypes: ")
display(df.dtypes)

# NOTE: print the missing values.
print("Missing value per columns: ")
display(df.isna().sum())

# NOTE: Double check the duplicated values from order_id columns
duplicate_order_id = df["order_id"].duplicated().sum()
print("Total duplicates in order_id: ", duplicate_order_id)


Dtypes: 


order_id            str
order_time          str
customer_id      object
city                str
total_amount    float64
currency            str
dtype: object

Missing value per columns: 


order_id        0
order_time      0
customer_id     1
city            1
total_amount    1
currency        1
dtype: int64

Total duplicates in order_id:  0


##### After scanning dataset, we can see data types are not correct, we have missing values from some columns

In [43]:
# NOTE: Change data types in order_time, customer_id
df["order_time"] = pd.to_datetime(df["order_time"], utc= True, errors="coerce")
df["customer_id"] = pd.to_numeric(df["customer_id"], errors="coerce")

# NOTE: Double check dtypes
display(df.dtypes)



order_id                        str
order_time      datetime64[us, UTC]
customer_id                 float64
city                            str
total_amount                float64
currency                        str
dtype: object

Cleaning rules (follow these exactly)

city: if null → replace with "UNKNOWN"

currency: if null → replace with "AUD"

total_amount: if null → replace with 0

customer_id: if null → replace with -1 (means “unknown customer”)

order_time: parse to datetime; if bad → keep as null (NaT)

In [44]:
# NOTE: Receiving instructions from manager with null value
df["city"] = df["city"].fillna("UNKNOWN")
df["currency"] = df["currency"].fillna("AUD")
df["total_amount"] = df["total_amount"].fillna(0)
df["customer_id"] = df["customer_id"].fillna(-1).astype("int64")

# NOTE: Parse timestamps; bad ones become NaT
df


,order_id,order_time,customer_id,city,total_amount,currency
0,ord_010,2026-02-05 09:05:10+00:00,201,Sydney,22.75,AUD
1,ord_011,2026-02-05 09:12:44+00:00,202,UNKNOWN,15.00,AUD
2,ord_012,2026-02-05 09:30:00+00:00,203,Brisbane,0.00,AUD
3,ord_013,NaT,204,Melbourne,9.50,AUD
4,ord_014,2026-02-05 10:05:55+00:00,-1,Perth,0.00,AUD


In [45]:
# NOTE: Double check NULL one more time
df.isna().sum()

order_id        0
order_time      1
customer_id     0
city            0
total_amount    0
currency        0
dtype: int64

#### Export to CSV file

In [46]:
output_path = Path("easy_orders_practice_clean.csv")
df.to_csv(output_path, index=False, encoding="utf-8")

print("Saved CSV: ", output_path.resolve())

Saved CSV:  C:\Users\pdinh\Python\Python_Full_Course\Python Practice\JSON-Python\easy_orders_practice_clean.csv
